In [1]:
import sys

sys.path.append('..')   

In [2]:
import pandas as pd
import numpy as np

from scarf import scarf

Scarf is not installed


In [4]:
scarf.fetch_dataset(
    dataset_name='kang_15K_pbmc_rnaseq',
    save_path='../data',
    as_zarr=True
)

scarf.fetch_dataset(
    dataset_name='kang_14K_ifnb-pbmc_rnaseq', 
    save_path='../data',
    as_zarr=True
)

In [3]:
ds_ctrl = scarf.DataStore(
    '../data/kang_15K_pbmc_rnaseq/data.zarr/',
    nthreads=4
)
ds_ctrl

DataStore has 8487 (14619) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'RNA_UMAP1', 'RNA_UMAP2', 
            'RNA_leiden_cluster', 'RNA_nCounts', 'RNA_nFeatures', 'RNA_percentMito', 'RNA_percentRibo', 
            'cluster_labels'
   RNA assay has 11352 (35635) features and following metadata:
            'I', 'ids', 'names', 'I__hvgs', 'dropOuts', 
            'nCells'

In [4]:
ds_stim = scarf.DataStore(
    '../data/kang_14K_ifnb-pbmc_rnaseq/data.zarr',
    nthreads=4
)
ds_stim

DataStore has 10111 (14446) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'RNA_UMAP1', 'RNA_UMAP2', 
            'RNA_leiden_cluster', 'RNA_nCounts', 'RNA_nFeatures', 'RNA_percentMito', 'RNA_percentRibo', 
            'cluster_labels'
   RNA assay has 11051 (35635) features and following metadata:
            'I', 'ids', 'names', 'I__hvgs', 'dropOuts', 
            'nCells'

In [5]:
#Can be used to merge multiple assays
scarf.ZarrMerge(
    # Path where merged Zarr files will be saved
    zarr_path='../data/kang_merged_pbmc_rnaseq.zarr',  
    
    # assays to be merged
    assays=[ds_ctrl.RNA, ds_stim.RNA],
    
    # these names will be preprended to the cell ids with '__' delimiter
    names=['ctrl', 'stim'],
    
    # Name of the merged assay. `overwrite` will remove an existing Zarr file.
    merge_assay_name='RNA',
    overwrite=True
).dump()

Writing data from assay 1/2 to merged file:   0%|                                                             …

Writing data from assay 2/2 to merged file:   0%|                                                             …

In [6]:
ds = scarf.DataStore(
    '../data/kang_merged_pbmc_rnaseq.zarr',
    nthreads=4
)

(RNA) Computing nCells and dropOuts:   0%|                                                                    …

(RNA) Computing nCounts:   0%|                                                                                …

(RNA) Computing nFeatures:   0%|                                                                              …

(RNA) Computing RNA_percentMito:   0%|                                                                        …

(RNA) Computing RNA_percentRibo:   0%|                                                                        …

In [7]:
df = ds.cells.to_pandas_dataframe(['ids','RNA_nCounts', 'orig_RNA_nCounts'])
df

,ids,RNA_nCounts,orig_RNA_nCounts
0,ctrl__TGTTAAGACACTGA-1,574.0,574.0
1,ctrl__TCACCCGAGCTATG-1,1682.0,1682.0
2,ctrl__TCACCCGAACCACA-1,1168.0,1168.0
3,ctrl__TGTAGTCTACCAGT-1,2158.0,2158.0
4,ctrl__TATCAGCTAGTAGA-1,1395.0,1395.0
...,...,...,...
29060,stim__TTGAACCTACGTAC-1,2941.0,2941.0
29061,stim__TTCCATGACCATAG-1,1620.0,1620.0
29062,stim__TTGAGGACTTTGTC-1,763.0,763.0
29063,stim__TTGAGGTGGTCACA-1,1049.0,1049.0


In [8]:
df[df['orig_RNA_nCounts'] != df['RNA_nCounts']]

,ids,RNA_nCounts,orig_RNA_nCounts


In [10]:
ds_ctrl.RNA.rawData.sum().compute()

23540929

In [11]:
ds_stim.RNA.rawData.sum().compute()

25059996

In [12]:
ds_ctrl.RNA.rawData.sum().compute() + ds_stim.RNA.rawData.sum().compute()

48600925

In [13]:
ds.RNA.rawData.sum().compute()

48600925